In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL
import warnings
import nbformat
warnings.filterwarnings("ignore")

df     = pd.read_csv("../data/processed/master_df.csv", index_col=0, parse_dates=True)
events = pd.read_csv("../data/raw/geopolitical_events.csv", parse_dates=["date"])

monthly = df["USDINR"].resample("ME").mean().dropna()

In [2]:
# ── STL DECOMPOSITION ───────────────────────────────────
stl    = STL(monthly, period=12, robust=True)
result = stl.fit()

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=["Observed USD/INR","Trend Component",
                    "Seasonal Component (12-month cycle)",
                    "Residual (unexplained — geopolitics lives here)"],
    vertical_spacing=0.08
)
components = {
    1: (monthly,          "#1A3A5C"),
    2: (result.trend,     "#2A9D8F"),
    3: (result.seasonal,  "#F4A261"),
    4: (result.resid,     "#D62828"),
}
for row, (series, color) in components.items():
    fig.add_trace(go.Scatter(
        x=series.index, y=series.values,
        mode="lines", line=dict(color=color, width=1.5),
        showlegend=False
    ), row=row, col=1)

# Mark geopolitical events on residual panel
for _, ev in events.iterrows():
    fig.add_vline(x=str(ev["date"]), line_dash="dot",
                  line_color="gray", line_width=0.8, row=4, col=1)

fig.update_layout(
    height=700, template="plotly_white",
    title="<b>STL Decomposition of USD/INR</b><br>"
          "<sub>Separating trend, seasonality, and geopolitical residuals</sub>"
)
fig.write_html("../data/processed/stl_decomposition.html")
fig.show()

In [3]:
# ── MONTHLY SEASONALITY HEATMAP ─────────────────────────
# Which month does INR typically weaken or strengthen?
monthly_ret = monthly.pct_change() * 100
heat_df = pd.DataFrame({
    "year":  monthly_ret.index.year,
    "month": monthly_ret.index.month,
    "ret":   monthly_ret.values
}).dropna()

pivot = heat_df.pivot_table(index="year", columns="month", values="ret", aggfunc="mean")
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                 "Jul","Aug","Sep","Oct","Nov","Dec"]

avg_by_month = heat_df.groupby("month")["ret"].mean()
months_full  = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Monthly INR Return Heatmap (% change, red=INR weakens)",
                    "Average INR Seasonality by Month (2015–2026)"],
    column_widths=[0.65, 0.35]
)

fig2.add_trace(go.Heatmap(
    z=pivot.values, x=list(pivot.columns), y=[str(y) for y in pivot.index],
    colorscale="RdBu_r", zmid=0,
    colorbar=dict(title="%", len=0.9, x=0.62),
    text=np.round(pivot.values,1), texttemplate="%{text}%",
    textfont=dict(size=8)
), row=1, col=1)

fig2.add_trace(go.Bar(
    x=months_full,
    y=avg_by_month.values,
    marker_color=["#D62828" if v > 0 else "#2A9D8F" for v in avg_by_month.values],
    showlegend=False
), row=1, col=2)

fig2.add_hline(y=0, line_dash="dash", line_color="black",
               line_width=1, row=1, col=2)
fig2.update_layout(height=420, template="plotly_white",
    title="<b>INR Seasonality Analysis</b><br>"
          "<sub>Red bars = INR typically weakens | Blue = INR typically strengthens</sub>")
fig2.write_html("../data/processed/seasonality_heatmap.html")
fig2.show()


In [4]:
# ── PRINT KEY SEASONAL INSIGHTS ─────────────────────────
print("\n=== SEASONAL INSIGHTS ===")
weakest  = months_full[avg_by_month.values.argmax()]
strongest= months_full[avg_by_month.values.argmin()]
print(f"INR historically weakest in:    {weakest}  ({avg_by_month.max():+.2f}% avg)")
print(f"INR historically strongest in:  {strongest} ({avg_by_month.min():+.2f}% avg)")
print("\nThese seasonal patterns become your baseline.")
print("Deviations from baseline = explained by macro or geopolitical shocks.")


=== SEASONAL INSIGHTS ===
INR historically weakest in:    Aug  (+0.69% avg)
INR historically strongest in:  Jan (-0.03% avg)

These seasonal patterns become your baseline.
Deviations from baseline = explained by macro or geopolitical shocks.
